In [2]:
import pandas as pd
import torch
from sklearn.utils import shuffle
import torch.nn as nn

In [3]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")  # you can continue going on here, like cuda:1 cuda:2....etc.
    print("Running on the GPU")
else:
    device = torch.device("cpu")
    print("Running on the CPU")

Running on the CPU


In [8]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "Sota"or cwd.name == "ML100K":
    chdir("/Users/haowen/Documents/Decentral RS/fed-learning-main")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code/Sota/ML100K
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [10]:
train = pd.read_csv("dataset/ml100k_train.csv")
test = pd.read_csv("dataset/ml100k_test.csv")
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [14]:
rating_matrix = train.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = rating_matrix.shape
# Scaling ratings to between 0 and 1, this helps our model by constraining predictions
min_rating, max_rating = train['rating'].min(), train['rating'].max()
rating_matrix = (rating_matrix - min_rating) / (max_rating - min_rating)
print(rating_matrix)
print(n_users*n_movies-rating_matrix.isnull().values.sum())

item_id  0     1     2     3     4     5     6     7     8     9     ...  \
user_id                                                              ...   
0        1.00   0.5  0.75  0.50   NaN   1.0   NaN   0.0   NaN  0.50  ...   
1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.25  ...   
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4        0.75   0.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
938       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   NaN  ...   
939       NaN   NaN   NaN  0.25   NaN   NaN  0.75   1.0   NaN   NaN  ...   
940       NaN   NaN   NaN   NaN   NaN   NaN  0.75   NaN   NaN   NaN  ...   
941       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
942       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

item_id  16

In [16]:
# Replacing missing ratings with -1 so we can filter them out later

rating_matrix[rating_matrix.isnull()] = -1
rating_matrix = torch.FloatTensor(rating_matrix.values)
print(rating_matrix)

tensor([[ 1.0000,  0.5000,  0.7500,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]])


In [18]:
# class GMF(torch.nn.Module):
#     def __init__(self, feature_size):
#         super().__init__()
#         self.fc = nn.Linear(feature_size, feature_size)
#         self.out = nn.Linear(feature_size, 1)
#         self.ReLU = nn.ReLU()

#     def forward(self, matrix, u_features, v_features):
#         x = torch.einsum('ik,jk->ijk',[u_features, v_features])
#         x = self.fc(x)
#         x = self.ReLU(x)
#         x = self.out(x)
#         predicted_rating = torch.sigmoid(x)
#         # non_zero_mask = (matrix != -1).type(torch.FloatTensor)
#         # predicted_rating = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
#         return predicted_rating

In [20]:
# A = torch.tensor([[1,2,3]])
# B = torch.tensor([[1,2,3]])
# print(torch.einsum('ik,jk->ijk',[A, B]))

In [22]:
class Fed_NCF(nn.Module):
    def __init__(self,user_num, item_num, factor_num):
        super(Fed_NCF,self).__init__()
        self.embed_user_GMF = nn.Embedding(user_num,factor_num)
        self.embed_item_GMF = nn.Embedding(item_num,factor_num)
        self.hidden_layer = nn.Linear(factor_num,factor_num)
        self.predict_layer = nn.Linear(factor_num,1)
        self._init_weight_()
        self.ReLU = nn.ReLU()
    def _init_weight_(self):
        nn.init.normal_(self.embed_item_GMF.weight,std=0.01)
        nn.init.normal_(self.embed_user_GMF.weight,std=0.01)

    def forward(self,user,item):
        embed_user_GMF = self.embed_user_GMF(user)
        embed_item_GMF = self.embed_item_GMF(item)
        output_GMF = torch.einsum('ik,jk->ijk',[embed_user_GMF, embed_item_GMF])
        # output_GMF = self.hidden_layer(output_GMF)
        # output_GMF = self.ReLU(output_GMF)
        prediction = torch.sigmoid(self.predict_layer(output_GMF))

        return torch.squeeze(prediction)

In [24]:
class GMFLoss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, predicted):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor).to(device)
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)
        
        return prediction_error

In [26]:
num_epoch = 100
latent_vectors = 20
gmf = Fed_NCF(n_users, n_movies, latent_vectors).to(device)
gmfloss = GMFLoss(lam_u=0.1, lam_v=0.1)
user = torch.LongTensor(range(n_users)).to(device)
item = torch.LongTensor(range(n_movies)).to(device)

optimizer = torch.optim.Adam(gmf.parameters(), lr=0.01)
for step, epoch in enumerate(range(num_epoch)):
    optimizer.zero_grad()
    # pred = gmf(torch.ones(n_users).long().to(device),torch.ones(n_movies).long().to(device))
    pred = gmf(user,item)
    loss = gmfloss(rating_matrix, pred)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"Step {step}, {loss:.3f}")

Step 0, 7466.214
Step 10, 6901.216
Step 20, 5969.574
Step 30, 4677.877
Step 40, 4067.388
Step 50, 3539.632
Step 60, 3113.248
Step 70, 2763.520
Step 80, 2472.319
Step 90, 2225.840


In [27]:
test_rating_matrix = test.pivot(index='user_id', columns='item_id', values='rating')

test_rating_matrix[test_rating_matrix.isnull()] = -1
test_rating_matrix = torch.FloatTensor(test_rating_matrix.values)
test_rating_matrix.shape

torch.Size([943, 1408])

In [30]:
train_items = train["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
test_items = test["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
# Create a mapping from item_id → index in train_items
item_to_index = pd.Series(train_items.index.values, index=train_items.values)

# Map test_items to train indices
test_indices_in_train = test_items.map(item_to_index)

print(test_indices_in_train)

0          0
1          1
2          2
3          3
4          4
        ... 
1403    1593
1404    1596
1405    1606
1406    1608
1407    1620
Name: item_id, Length: 1408, dtype: int64


In [32]:
## test FedNCF
non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor).to(device)
num = torch.sum(non_zero_mask)
user = torch.LongTensor(range(n_users)).to(device)
item = torch.LongTensor(range(n_movies)).to(device)
predicted_ratings = gmf(user, item)
predicted_ratings = predicted_ratings[:,test_indices_in_train]
predicted_ratings = (predicted_ratings*(max_rating - min_rating) + min_rating)*non_zero_mask
actual_ratings = test_rating_matrix*non_zero_mask

AE_diff = torch.abs(predicted_ratings - actual_ratings)
SE_diff = (predicted_ratings - actual_ratings)**2

test_MAE = torch.sum(AE_diff)/num
test_RMSE = torch.sqrt(torch.sum(SE_diff)/num)
print('test_MAE =', test_MAE.data.cpu().numpy())
print('test_RMSE =', test_RMSE.data.cpu().numpy())

test_MAE = 0.7821616
test_RMSE = 1.0108749
